# Consensus, plainly: Raft & Paxos — companion notebook

Season 2, Part 2 of *System Design, Plainly*. Every number in the post is recomputed here. Run all cells top to bottom.

In [ ]:
import math

def quorum_loss(n, p):
    """P(cluster loses quorum): at least n - quorum + 1 nodes down,
    where quorum = floor(n/2) + 1."""
    k_min = n - n // 2  # ceil(n/2)
    return sum(math.comb(n, k) * p**k * (1 - p)**(n - k)
               for k in range(k_min, n + 1))

for p in (0.01, 0.05):
    print(f'--- per-node downtime {p:.0%} ---')
    for n in (1, 3, 5, 7):
        q = quorum_loss(n, p)
        print(f'n={n}: quorum-loss prob {q:.3e}  = {q*525600:,.1f} min/yr')

# each extra pair of nodes buys ~30x at p=1%
assert abs(quorum_loss(3, 0.01) - 2.98e-4) < 1e-5
assert abs(quorum_loss(5, 0.01) - 9.85e-6) < 1e-6
assert abs(quorum_loss(7, 0.01) - 3.40e-7) < 1e-7
assert quorum_loss(3, 0.01) / quorum_loss(5, 0.01) > 25  # ~30x step

In [ ]:
import random

def dueling_rounds(k, trials=20000, seed=11):
    """Classic Paxos: k proposers draw random start times each round;
    a round commits only if the earliest finishes (0.25 of the window)
    before the second-earliest starts — otherwise it is pre-empted."""
    rng = random.Random(seed)
    total = 0
    for _ in range(trials):
        rounds = 0
        while True:
            rounds += 1
            starts = sorted(rng.random() for _ in range(k))
            if k == 1 or starts[1] - starts[0] > 0.25:
                break
        total += rounds
    return total / trials

for k in (1, 2, 3, 4, 5, 6):
    print(f'{k} proposer(s): {dueling_rounds(k):.2f} mean rounds to commit')
print('Raft / Multi-Paxos with one stable leader: 1.00 rounds, always')

assert dueling_rounds(1) == 1.0
assert 1.6 < dueling_rounds(2) < 2.0
assert dueling_rounds(6) > 4.0

In [ ]:
import math
import numpy as np

def commit_latency(n, rtt_med_ms, trials=20000, seed=23):
    """Median commit latency = RTT to the (quorum-1)th fastest follower."""
    rng = np.random.default_rng(seed)
    q = n // 2 + 1
    samples = rng.lognormal(mean=math.log(rtt_med_ms), sigma=0.35,
                            size=(trials, n - 1))
    samples.sort(axis=1)
    return float(np.median(samples[:, q - 2]))

for name, med in [('same-DC', 0.5), ('cross-AZ', 2.0), ('cross-region', 80.0)]:
    row = {n: round(commit_latency(n, med), 2) for n in (3, 5, 7)}
    print(f'{name:13s} median commit latency (ms): {row}')

assert commit_latency(3, 0.5) < 1.0
assert commit_latency(7, 80.0) > 60
assert commit_latency(7, 0.5) > commit_latency(3, 0.5)  # bigger quorum, slower tail

In [ ]:
"""Raft log replication toy: an entry commits the moment a majority stores it.
A 2-node minority partition can never commit — that is the safety guarantee."""
import random

def replicate(n=5, seed=5):
    rng = random.Random(seed)
    quorum = n // 2 + 1
    acks = sorted(rng.uniform(0.2, 1.5) for _ in range(n - 1))
    commit_at = acks[quorum - 2]  # (quorum-1)th follower ack
    return acks, commit_at

acks, commit_at = replicate()
print(f'follower acks (ms): {[round(a, 2) for a in acks]}')
print(f'entry commits at {commit_at:.2f} ms — the 3rd ack, a majority of 5')
assert commit_at == sorted(acks)[1]

# minority partition: 2 nodes of 5 can never reach quorum 3
minority = 2
print(f'minority partition holds {minority} nodes, quorum is 3: '
      f'can commit? {minority >= 3}')
assert minority < 3

# even cluster sizes buy nothing: 4 nodes tolerate 1 failure, same as 3
for n in (3, 4, 5, 6):
    print(f'n={n}: tolerates {(n - 1) // 2} failure(s)')
assert (4 - 1) // 2 == (3 - 1) // 2 == 1
assert (6 - 1) // 2 == (5 - 1) // 2 == 2